# Part of speech 
This page examines the performance of DaCy and competing Danish models in the task of part-of-speech tagging. The general accuracy is evaluated on two datasets, dacy data and hist-pos-dk. Apart from accuracy, the evaluation includes the models' robustness and checks for biases.

# Historical data
To compare the models performance on historical data, we used the [hist-pos-dk](https://huggingface.co/datasets/chcaa/hist-dk-pos) dataset. It contains example sentences extracted from a piece of historical Danish newspaper fiction published in 1824, which include archaic Danish spelling (eg....). 

In [ ]:
# i have to use the utils
# testing if datasets load - they do
import sys; sys.path.insert(0, "/work/JohanaMayerova#0832/DaCy/docs")
import spacy
from spacy.scorer import Scorer
from spacy.tokens import Doc
from spacy.training import Example
from evaluation.datasets import datasets
from evaluation.models import MODELS

# datasets are catalogue-registered:
dacy_data = datasets.get("dacy_data")()        # {"train": [...], "dev": [...], "test": [...]}
hist_pos  = datasets.get("hist_pos_dk")()      # {"train": [...]} - only split available

nlp = MODELS["da_dacy_edge-0.1.0"]()

def evaluate_pos(gold_examples: list[Example]) -> dict:
    # run model on pre-segmented tokens (avoids retokenization mismatch)
    word_docs = [Doc(nlp.vocab, words=[t.text for t in ex.reference]) for ex in gold_examples]
    pred_docs = list(nlp.pipe(word_docs, batch_size=32))
    examples = [Example(pred, ex.reference) for pred, ex in zip(pred_docs, gold_examples)]
    return Scorer.score_token_attr(examples, "pos")

scores_hist = evaluate_pos(hist_pos["train"])   # hist only has a "train" split
print("hist-pos-dk  POS acc:", scores_hist["pos_acc"])

scores_dacy = evaluate_pos(dacy_data["test"])
print("dacy-data    POS acc:", scores_dacy["pos_acc"])

hist-pos-dk  POS acc: 0.875
dacy-data    POS acc: 0.9472067856098274
